# 🏦 financial_services assets: produce + evaluate

One notebook to GENERATE and VET the financial_services asset bank (keeps the top-level `notebooks/` focused on simulate/eval/extract). Cells drive the CLI, then load the outputs for preview + quality assessment.

**RUNBOOK** (the generate + vet gate; needs `NVIDIA_API_KEY` + network for the generate/evaluate cells):
1. 🏗️ Generate → 2. ✅ Validate (deterministic gate) → 3. ⚖️ Evaluate (DD-judge scorecard) → 4. 👀 Preview + assess quality → iterate on prompts/knobs → freeze (commit `assets/financial_services/<locale>/`).

**Switching locale**: set `LOCALE` in the config cell: every path below derives from it, and the cell prints which locales have an authored `region_spec` and which already have a bank. For a non-Latin locale the deterministic gate's `language_script` check confirms the corpus really came out in that script (the most likely failure when the doc-gen model ignores the language instruction).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "asset_gen").is_dir())
SPEC_DIR = REPO_ROOT / "asset_gen/financial_services/region_spec"

# ── The knob: which locale's bank are we producing? ──────────────────────────
LOCALE = "hi_Deva_IN"

# ── Knobs: generation size ───────────────────────────────────────────────────
# Used by BOTH the dry-run and the live cell below, so the plan you preview is the
# plan that actually runs. Both mirror the pipeline defaults and are passed on the
# command line, so a run records what it used -- change them here to experiment.
#
# DOCS_PER_PRODUCT = 40 is the COVERAGE default. Each product's plan round-robins
# across 12 genres, so 40 gives every expandable genre 5 slots: how_to_guide and
# troubleshooting then carry all their intents (opening, closing, a failed action --
# what the dynamic tier asks about) WHILE faq still keeps a per-variable explainer
# for every authoritative value. Tuning it:
#   ~10   fast smoke regen; thin corpus, retrieval is barely a challenge.
#   < 17  the relationship angles stop fitting for every product
#         (`required documentation`, `cancel, close, or downgrade`) -- the gap that
#         left en_IN with no answer to "which documents count as proof of address".
#   > 46  a product starts running out of DISTINCT angles (supply is
#         40 + 3 x n_variables, so 46-52 in en_US and 49-58 in en_IN). Past its own
#         supply a product's plan returns fewer docs than asked and logs, rather
#         than repeating an angle.
#
# MAX_DOCS_PER_CLUSTER = 10: each doc_set call emits that many docs as ONE
# structured object, and bigger clusters overwhelmed the hub (5xx) and dropped whole
# clusters. Coherence does not need big clusters -- the shared brief, the
# deterministic angle plan and the whole-institution sibling index carry it across
# batches.
DOCS_PER_PRODUCT = 40
MAX_DOCS_PER_CLUSTER = 10

BANK_DIR = REPO_ROOT / "output" / f"finance_{LOCALE}"              # scratch --out
ASSETS_DIR = REPO_ROOT / "assets" / "financial_services" / LOCALE  # canonical
REGION_SPEC = SPEC_DIR / f"{LOCALE}.yaml"

# Resolve the CLI to THIS repo's environment so `!usersim` cells never hit a stale
# global install on the shell PATH (which can miss newer flags like --max-parallel).
_usersim_candidates = [REPO_ROOT / ".venv" / "bin" / "usersim",
                    Path(sys.executable).with_name("usersim")]
USERSIM = next((str(c) for c in _usersim_candidates if c.exists()), "usersim")

print("repo:", REPO_ROOT, "\nusersim:", USERSIM)
print(f"\nLOCALE={LOCALE!r}  docs_per_product={DOCS_PER_PRODUCT}  "
      f"max_docs_per_cluster={MAX_DOCS_PER_CLUSTER}")
print("  scratch:", BANK_DIR)
print("  frozen :", ASSETS_DIR)

# Which locales can be produced, and what already exists for each.
print(f"\n{'locale':<14}{'region_spec':<13}{'scratch':<10}frozen")
for _spec in sorted(SPEC_DIR.glob("*.yaml")):
    _loc = _spec.stem
    _scratch = REPO_ROOT / "output" / f"finance_{_loc}"
    _frozen = REPO_ROOT / "assets" / "financial_services" / _loc
    _n = len(list(_frozen.rglob("corpus.yaml"))) if _frozen.is_dir() else 0
    print(f"{_loc:<14}{'yes':<13}{('yes' if _scratch.is_dir() else '-'):<10}"
          f"{str(_n) + ' institution(s)' if _n else '-'}"
          f"{'   <- selected' if _loc == LOCALE else ''}")

if not REGION_SPEC.exists():
    print(f"\n\u26a0\ufe0f  no region_spec at {REGION_SPEC} -- generation needs one.")
    print("   A language variant can inherit an existing region's model with")
    print("   `extends: <base-locale>` and override only what differs.")
    print("   Note hi_Latn_IN is DERIVED from hi_Deva_IN by transliteration")
    print("   (`romanize-assets`), so it is not generated in this notebook.")
else:
    print(f"\nregion_spec: {REGION_SPEC.name} \u2705")


## 🔑 Credentials

The generate/evaluate cells hit the inference endpoint named in your models config: doc-gen, judge, and embedding all authenticate with `NVIDIA_API_KEY`. The first run prompts via `getpass` (no echo); later runs see the key in `os.environ` and pass through to the `!usersim` subprocess. `prompt_for_provider_keys` walks the models config and asks only for what's missing.

In [ ]:
import getpass
import os

from cli._models import default_models_path, load_models_config, prompt_for_provider_keys

prompt_for_provider_keys(load_models_config(default_models_path()))
# doc-gen + embedding + judge providers all key on NVIDIA_API_KEY; ensure it's set.
if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA API key: ")
print("🔑 NVIDIA_API_KEY set" if os.environ.get("NVIDIA_API_KEY") else "⚠️  no key set")

## ⚙️ Model knobs

Swap which model each asset-gen alias uses. Provider auto-routes via `cli/model_catalog.py` (hub-served models → `nvidia_inference_hub`). These become `--model ALIAS=MODEL` flags on the gen/evaluate CLI calls below; defaults mirror `cli/models_default.toml`, so edit a value to override for this run only.

In [ ]:
MODEL_OVERRIDES = {
    # NOTE: doc_gen_model AND asset_judge_model are pinned in cli/models_default.toml
    # (doc_gen = gemma-4-31b-it @ max_tokens=16384; judge = gpt-5.4 @ temperature=0).
    # Do NOT bare-override them here (e.g. `--model doc_gen_model=...`): a bare
    # `--model alias=model` RE-RESOLVES sampling from the catalog, which DISCARDS the
    # TOML row's per-alias knobs -- it resets doc-gen max_tokens to the catalog's 8192
    # (truncating large clusters mid-JSON) and the judge's temperature to 1.0 (noisy
    # scores). Change the model or its knobs in the TOML instead.
    # dense retrieval vectors -- pick ONE embedding model:
    # "embedding_model":   "nvidia/nvidia/nemotron-3-embed-1b",  # NVIDIA nemotron-3-embed
    "embedding_model":   "nvidia/qwen/qwen3-embedding-0.6b",   # Qwen3 (input_type=passage, truncate=NONE via catalog)
}
# Concurrency for doc generation. Lower it if the hub throws 5xx/timeouts under load
# (each doc_set call is a heavy structured generation); raise it for faster throughput.
DOC_GEN_CONCURRENCY = 6

MODEL_FLAGS = " ".join(f"--model {a}={m}" for a, m in MODEL_OVERRIDES.items())
MODEL_FLAGS += f" --max-parallel doc_gen_model={DOC_GEN_CONCURRENCY}"
print("model flags:", MODEL_FLAGS)

### 🏗️ 1. Generate (CLI; needs `NVIDIA_API_KEY` + network)

In [ ]:
# Preview the plan first (offline, no LLM). Sizes come from the config cell, so
# this preview and the live run below can never drift apart.
!cd {REPO_ROOT} && {USERSIM} gen-assets --domain financial_services --locale {LOCALE} --docs-per-product {DOCS_PER_PRODUCT} --max-docs-per-cluster {MAX_DOCS_PER_CLUSTER} --dry-run

In [ ]:
# Live run ({MODEL_FLAGS} pins doc-gen / judge / embedding models -- see the knobs cell)
!cd {REPO_ROOT} && {USERSIM} gen-assets --domain financial_services --locale {LOCALE} --docs-per-product {DOCS_PER_PRODUCT} --max-docs-per-cluster {MAX_DOCS_PER_CLUSTER} --out {BANK_DIR} {MODEL_FLAGS}

### ✅ 2. Validate (deterministic gate)

In [ ]:
!cd {REPO_ROOT} && {USERSIM} validate-assets --domain financial_services --locale {LOCALE} --dir {BANK_DIR}

### ⚖️ 3. Evaluate (DD-judge scorecard)

Non-blocking LLM-judge quality scorecard (needs the judge model + network). Complements the deterministic validate gate above.

In [ ]:
!cd {REPO_ROOT} && {USERSIM} evaluate-assets --domain financial_services --locale {LOCALE} --dir {BANK_DIR} {MODEL_FLAGS}

### 👀 4a. Preview: institution brief + docs side-by-side with authoritative values
Eyeball numeric faithfulness (do the docs state the spec's exact values?) and voice/positioning.

In [ ]:
import json, yaml
from asset_gen.financial_services.spec import load_region_spec

# Defined here and reused by the quality cell below.
spec = load_region_spec(REGION_SPEC) if REGION_SPEC.exists() else None
prod_values, prod_local = {}, {}
if spec is not None:
    prod_values = {(i.id, p.id): {k: v.value for k, v in p.variables.items()}
                   for i in spec.institutions for p in i.products}
    prod_local = {(i.id, p.id): p.display_name_local
                  for i in spec.institutions for p in i.products}
else:
    print(f"(no region_spec for {LOCALE} -- authoritative values unavailable; "
          f"showing documents only)")

gr = {}
gr_path = BANK_DIR / "_generation_report.json"
if gr_path.exists():
    gr = json.loads(gr_path.read_text()).get("institutions", {})

if not BANK_DIR.is_dir():
    print(f"no bank at {BANK_DIR} -- run Generate first")

# Institutions are nested under a type dir (<type>/<id>/); walk corpus.yaml files.
for corpus_path in (sorted(BANK_DIR.rglob("corpus.yaml")) if BANK_DIR.is_dir() else []):
    idir = corpus_path.parent
    iid = idir.name
    brief = (gr.get(iid) or {}).get("brief") or {}
    meta = yaml.safe_load((idir / "institution_meta.yaml").read_text()) or {}
    local = meta.get("display_name_local")
    print(f"\n===== {iid}{f' ({local})' if local else ''} =====")
    if brief:
        print("  positioning:", brief.get("positioning", ""))
        print("  voice:", brief.get("brand_voice_guide", ""))
    docs = (yaml.safe_load(corpus_path.read_text()) or {}).get("documents", [])
    for d in docs[:3]:
        key = (iid, d.get("product_category"))
        print(f"\n  [{d.get('document_type')}] {d.get('title')}")
        if prod_values.get(key):
            print(f"    authoritative values: {prod_values[key]}")
        # For a non-Latin locale the docs should use the AUTHORED local product
        # name; a doc that invented its own translation shows up here.
        if prod_local.get(key):
            body_and_title = f"{d.get('title','')} {d.get('body','')}"
            hit = "\u2705" if prod_local[key] in body_and_title else "\u26a0\ufe0f not used"
            print(f"    local product name: {prod_local[key]}  {hit}")
        print(f"    body: {d.get('body','')[:240]}")


### 📊 4b. Quality assessment: validate report + eval scorecard + per-cluster QC

In [ ]:
from asset_gen.financial_services.validate import validate_bank

# Re-derive if this cell is run without the preview cell above.
if "spec" not in dir():
    from asset_gen.financial_services.spec import load_region_spec
    spec = load_region_spec(REGION_SPEC) if REGION_SPEC.exists() else None
if "gr" not in dir():
    import json as _json
    _p = BANK_DIR / "_generation_report.json"
    gr = _json.loads(_p.read_text()).get("institutions", {}) if _p.exists() else {}

print(validate_bank(BANK_DIR, spec).to_text())

sc = BANK_DIR / "_eval_scorecard.parquet"
if sc.exists():
    import pandas as pd
    df = pd.read_parquet(sc)
    axis_cols = [c for c in df.columns if c.endswith("_score")]
    print("\nDD-judge scorecard (per-doc axes, 1-5):")
    print(df[axis_cols].mean().round(2).to_string())
    # Calibration note: realism / clarity_readability / self_containment have not
    # been observed to reach 5, so their means run ~2.6-3.7 on a healthy corpus --
    # read them against the en_US baseline rather than against 5. Triage at <=2:
    # a <3 cut flags roughly half the corpus and is not a queue you can work.
    #
    # How much of a change is real? The judge is a reasoning model, so it is NOT
    # reproducible even at temperature 0 (see cli/models_default.toml). Re-judging one
    # UNCHANGED corpus moved 16-24% of per-axis scores by a full point and shifted
    # per-genre triage by 5.4pt on average, 20pt at worst. Practical thresholds:
    #   corpus-wide mean  -- trustworthy (re-judge churn is +/-0.014)
    #   per-genre mean    -- believe a shift >= ~0.2 (re-judge sd is 0.07)
    #   per-genre triage  -- indicative only; it is the noisiest number here
    # Deterministic checks (validate_bank, deflection/leak greps) carry no judge
    # noise at all, so prefer them whenever the property can be measured directly.
    low = df[df[axis_cols].min(axis=1) <= 2]
    print(f"\n{len(low)}/{len(df)} doc(s) with any axis <=2 (worst first):")
    low = low.assign(_min=low[axis_cols].min(axis=1)).sort_values("_min")
    print(low[["id", "document_type", *axis_cols]].head(15).to_string(index=False))
else:
    print("\n(no _eval_scorecard.parquet yet -- run the evaluate cell above)")

# Generation-time cluster QC -- note the DIFFERENT scale from the per-doc scorecard
# above: these rubrics run 0-4 and 4 is the TOP score (see pipeline.docset_qc_scores).
# So faithfulness pinned at a flat 4.0 across every cluster is the pass state, not a
# dead metric -- the deterministic numeric_faithfulness check in the gate above is an
# independent cross-check on the same property. Cohesion/distinctness DO vary, and
# ~3.4-3.6 out of 4 is the healthy band observed so far.
if gr:
    print("\nGeneration-time cluster QC (rubrics are 0-4; 4 = best):")
for iid, idata in gr.items():
    for c in idata.get("clusters", []):
        print(iid, c.get("cluster_kind"),
              "faithfulness=", c.get("numeric_faithfulness"),
              "cohesion=", c.get("cohesion"), "distinctness=", c.get("distinctness"))


## 🧊 5. Freeze: promote the vetted bank into `assets/`

Generation writes to the **scratch** `BANK_DIR` (`output/finance_<locale>/`) so re-runs never clobber the committed bank. The probe loads from the **canonical** `assets/financial_services/<locale>/`. Once the vet gate + review look good, promote the scratch bank there, then `git diff` and commit.

Notes:
- Curated `tasks.yaml` files are **preserved** (never overwritten): `gen-assets` only writes corpus/tools/meta/embeddings.
- Running `usersim gen-assets ... ` **without** `--out` writes straight to `assets/financial_services/<locale>/` (skipping the scratch dir): use that once you trust the pipeline.

In [ ]:
import shutil

FREEZE = True  # flip to True after the vet gate + review look good
_PROMOTE = ("institution_meta.yaml", "corpus.yaml", "tools.yaml", "embeddings.parquet")

if not FREEZE:
    print(f"FREEZE=False: set True to promote the vetted bank:\n"
          f"  {BANK_DIR}\n  ->  {ASSETS_DIR}\n"
          f"(region_meta + per-institution corpus/tools/meta/embeddings are overwritten; "
          f"hand-authored tasks.yaml / dynamic.yaml are preserved)")
elif not BANK_DIR.is_dir():
    print(f"nothing to freeze: {BANK_DIR} does not exist: run Generate first")
else:
    ASSETS_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BANK_DIR / "region_meta.yaml", ASSETS_DIR / "region_meta.yaml")
    # Provenance has to travel with the corpus it describes. The vet gate reads
    # per-cluster QC out of this file, and the evaluator reads each institution's
    # GENERATED brief out of it -- so a frozen bank without it is judged for
    # brief-consistency against an empty brief, and the gate can only report
    # "provenance/QC unknown".
    _report = BANK_DIR / "_generation_report.json"
    if _report.exists():
        shutil.copy2(_report, ASSETS_DIR / "_generation_report.json")
    frozen, untasked = [], []
    # Institutions are nested under a type dir (<type>/<id>/); mirror that structure.
    for meta_path in sorted(BANK_DIR.rglob("institution_meta.yaml")):
        rel = meta_path.parent.relative_to(BANK_DIR)   # e.g. bank/northwind_bank
        dst = ASSETS_DIR / rel
        dst.mkdir(parents=True, exist_ok=True)
        for fname in _PROMOTE:
            src = meta_path.parent / fname
            if src.exists():
                shutil.copy2(src, dst / fname)
        frozen.append(str(rel))
        if not (dst / "tasks.yaml").exists():
            untasked.append(str(rel))
    print(f"\u2705 froze {len(frozen)} institution(s) -> {ASSETS_DIR}\n   {frozen}")

    # Integrity check: re-validate the FROZEN bank WITH its tasks.yaml, which
    # catches tasks<->corpus drift (dangling gold_document_ids after a
    # regeneration) BEFORE you commit. The scratch --out dir is corpus-only, so
    # this can only run here.
    if untasked:
        # A NEW locale has no curated tasks yet -- that is the expected state
        # until they are authored, NOT drift. Don't cry wolf on the first freeze.
        print(f"\n   \u2139\ufe0f  corpus-only so far: {len(untasked)} institution(s) have no "
              f"tasks.yaml yet\n      {untasked}")
        print("      Expected for a newly generated locale. Author tasks.yaml per "
              "institution\n      (and a locale-level dynamic.yaml) next; the "
              "tasks<->corpus integrity\n      check below starts applying once they exist.")
    else:
        from usersim.engine.core.finance_bank import (
            FinanceBankError, load_finance_bank, reset_finance_bank_cache)
        reset_finance_bank_cache()   # don't validate a stale cached bank
        try:
            bank = load_finance_bank(ASSETS_DIR)
            n_tasks = sum(len(i.templates) for i in bank.institutions)
            print(f"   \u2705 bank loads: {len(bank.institutions)} institution(s), "
                  f"{n_tasks} task template(s) (tasks<->corpus integrity OK).")
            print("      Review `git diff` then commit.")
        except FinanceBankError as e:
            print(f"   \u26d4 FROZEN BANK DOES NOT LOAD -- fix before committing:\n   {e}")
